<a href="https://colab.research.google.com/github/TonicAI/cookbooks/blob/main/sft/healthcare/textual_fireworks_fine_tune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Safely fine-tuning with sensitive data

Fine-tuning models with sensitive data presents additional challenges beyond model convergence. If we fine-tune on unredacted PII/PHI, then that sensitive information is potentially memorized by the model. The model artifact itself becomes sensitive, constraining where it can be deployed and who can query it. On the other hand, simply removing PII/PHI from the data may destroy its utility, particularly when that data is relevant to the fine-tuning problem. The ideal solution is to _redact & synthesize_ the sensitive information in a way that preserves its utility.

In this notebook, we'll show how to use [Tonic Textual](https://tonic.ai/textual) to de-identify sensitive data by swapping every name, date, address and other PII for realistic, fake values that are consistent within each record. The result is a pipeline where no sensitive data embeds in model weights.

Our example task is structured extraction: turning free-text encounter notes into typed records with HL7-aligned value sets. Frontier models can zero-shot this task, but operating at scale with frontier prices is prohibitively expensive. A small language model fine-tuned for the task delivers the same structured output at a fraction of the cost.

Does synthesis destroy the training signal? That's an empirical question, so we measure it. We fine-tune the same base model twice on [Fireworks](https://fireworks.ai) — once on synthesized notes, once on the originals — and evaluate both, plus the un-tuned base model, on real held-out notes:

| contender | trains on | shows |
|---|---|---|
| base model, zero-shot | — | why fine-tune at all |
| fine-tuned on **synthetic** | de-identified notes | the safe pipeline |
| fine-tuned on **real** | original notes | what synthesis costs us |

If the last two rows come out close, de-identification is a free lunch.

## The dataset

Real clinical notes can't be published, so this demo uses [TonicAI/synthetic_clinical_notes](https://huggingface.co/datasets/TonicAI/synthetic_clinical_notes) and *role-plays* it as PHI. It was built from [Synthea](https://synthetichealth.github.io/synthea/) synthetic patient records, with notes generated by a modified [chatty-notes](https://github.com/synthetichealth/chatty-notes/tree/main). Each row pairs a free-text `note` with ground-truth structured `encounter_data`.

> Jason Walonoski et al., *Synthea: An approach, method, and software mechanism for generating synthetic patients and the synthetic electronic health care record*, JAMIA, Volume 25, Issue 3, March 2018, Pages 230–238, https://doi.org/10.1093/jamia/ocx079

## Setup

You'll need:

- `TONIC_TEXTUAL_API_KEY` — from [Tonic Textual](https://textual.tonic.ai)
- `FIREWORKS_API_KEY` and your account id — from [fireworks.ai](https://fireworks.ai)

A cost note before running end-to-end: the two fine-tuning jobs bill per training token — a few dollars at the defaults here, plus 10–20 minutes of waiting — and the evaluation uses a dedicated GPU deployment billed per GPU-second while it exists (the cleanup cell at the end deletes it).

**Running on Colab:** click the badge above, then add `TONIC_TEXTUAL_API_KEY`, `FIREWORKS_API_KEY`, and `FIREWORKS_ACCOUNT_ID` in the Secrets panel (the 🔑 key icon in the left sidebar) and grant this notebook access. Running locally, export them as environment variables instead — the next cell handles both. No GPU runtime is needed: every heavy step (de-identification, fine-tuning, evaluation) runs on Tonic and Fireworks infrastructure, so the default CPU runtime is fine.

In [1]:
%pip install -q --pre fireworks-ai tonic-textual datasets pandas tqdm

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import json
import os
import time
import concurrent.futures as cf
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

from fireworks import Fireworks, NotFoundError
from tonic_textual.redact_api import TextualNer
from tonic_textual.classes.generator_metadata.name_generator_metadata import NameGeneratorMetadata

# Credentials — works on Colab (Secrets panel) and locally (environment variables).
def load_secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:  # Colab: read from the Secrets panel (key icon in the left sidebar)
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:  # anywhere else: prompt once, without echoing
        from getpass import getpass
        value = getpass(f"{name}: ")
    os.environ[name] = value
    return value

load_secret("TONIC_TEXTUAL_API_KEY")
load_secret("FIREWORKS_API_KEY")
FIREWORKS_ACCOUNT = load_secret("FIREWORKS_ACCOUNT_ID")

textual = TextualNer(api_key=os.environ["TONIC_TEXTUAL_API_KEY"])
fw = Fireworks(account_id=FIREWORKS_ACCOUNT)

BASE_MODEL = "accounts/fireworks/models/llama-v3p2-3b-instruct"

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

## 1. Load the clinical notes

In [3]:
ds = load_dataset("TonicAI/synthetic_clinical_notes")
ds

DatasetDict({
    train: Dataset({
        features: ['note', 'encounter_data'],
        num_rows: 1028
    })
    validation: Dataset({
        features: ['note', 'encounter_data'],
        num_rows: 672
    })
    test: Dataset({
        features: ['note', 'encounter_data'],
        num_rows: 1681
    })
})

In [4]:
example = ds["train"][0]
print(example["note"][:1000], "…")
print("\n--- ground-truth encounter_data ---")
print(json.dumps(example["encounter_data"], indent=2))

**Medical Note**

**Patient Name:** Harvey D'Amore  
**Date of Birth:** September 12, 1964  
**Medical Record Number:** HD-AM-0901964  
**Date of Encounter:** October 18, 2023  
**Attending Dentist:** Dr. Emily Thompson  
**Location:** Greenfield Dental Clinic, 123 Oak Street, Springfield, IL  

**Reason for Visit:**  
Mr. Harvey D'Amore presents for a routine check-up and management of gingivitis.

**Chief Complaint:**  
The patient reports experiencing swollen and tender gums over the past few weeks, with occasional bleeding during brushing and persistent bad breath.

**History of Present Illness:**  
Mr. D'Amore is a 59-year-old male who has been experiencing gingival tenderness and bleeding upon brushing for approximately three weeks. He reports that bleeding is most notable when flossing and notes some persistent bad breath, which has not improved with regular oral hygiene measures. Mr. D'Amore admits to occasionally skipping his night-time oral hygiene routine. He denies any rece

In [5]:
response = textual.redact(
	'**Medical Note**  **Patient Name:** Harvey D\'Amore  \n   D’Amore is a 59-year-old male who has been experiencing gingival tenderness and bleeding upon brushing for approximately three weeks. He reports that bleeding is most notable when flossing and notes some persistent bad breath, which has not improved with regular oral hygiene measures. Mr. D’Amore admits to occasionally skipping his night-time oral hygiene routine. He denies any rece …',
	generator_config={'NAME_FAMILY': 'Synthesis','NAME_GIVEN': 'Synthesis'}, generator_default='Off'
)
response.de_identify_results

[{'start': 36,
  'end': 42,
  'new_start': 36,
  'new_end': 42,
  'label': 'NAME_GIVEN',
  'text': 'Harvey',
  'score': 0.96484375,
  'language': 'xx',
  'new_text': 'Lorean'},
 {'start': 43,
  'end': 50,
  'new_start': 43,
  'new_end': 50,
  'label': 'NAME_FAMILY',
  'text': "D'Amore",
  'score': 0.983393132686615,
  'language': 'xx',
  'new_text': 'Bennice'},
 {'start': 56,
  'end': 63,
  'new_start': 56,
  'new_end': 66,
  'label': 'NAME_FAMILY',
  'text': 'D’Amore',
  'score': 0.978119969367981,
  'language': 'xx',
  'new_text': 'Challender'},
 {'start': 346,
  'end': 353,
  'new_start': 349,
  'new_end': 359,
  'label': 'NAME_FAMILY',
  'text': 'D’Amore',
  'score': 0.982029378414154,
  'language': 'xx',
  'new_text': 'Challender'}]

## 2. De-identify with Tonic Textual

[Textual](https://docs.tonic.ai/textual) detects PHI spans — names, dates, addresses, phone numbers, IDs — and replaces them. Two configuration choices matter for fine-tuning:

- **Synthesis, not redaction.** Replacing "Harvey D'Amore" with a `[NAME_GIVEN]` token would leave unnatural text and destroy our extraction labels. Synthesis swaps in a realistic substitute, so the notes still read like notes and the task is unchanged.
- **Consistency between note and labels.** Each `{note, encounter_data}` record goes through `redact_json` as a single unit with a fixed per-record seed, so the same synthetic name lands in the note text *and* in the ground-truth JSON. The input→output mapping the model has to learn survives de-identification intact.

Everything that isn't PHI — symptoms, procedures, medications, clinical reasoning — passes through untouched.

In [6]:
PHI_LABELS = [
    "NAME_GIVEN",
    "NAME_FAMILY",
    "LOCATION_ADDRESS",
    "LOCATION_STATE",
    "LOCATION_CITY",
    "LOCATION_ZIP",
    "LOCATION_COUNTRY",
    "PHONE_NUMBER",
    "EMAIL_ADDRESS",
    "CREDIT_CARD",
    "CC_EXP",
    "CVV",
    "MONEY",
    "ORGANIZATION",
    "DOB",
    "DATE_TIME",
    "URL",
    "NUMERIC_PII",
    "HEALTHCARE_ID",
]

DEID_CONFIG = dict(
    generator_config={label: "Synthesis" for label in PHI_LABELS},
    # keep synthetic given names gender-consistent — gender is an extraction target
    generator_metadata={"NAME_GIVEN": NameGeneratorMetadata(preserve_gender=True)},
    generator_default="Off",
)

In [7]:
response = textual.redact_json(example, random_seed=42, **DEID_CONFIG)
synthetic_example = json.loads(response.redacted_text)

print("original name :", example["encounter_data"]["name"])
print("synthetic name:", synthetic_example["encounter_data"]["name"])
print()
print(synthetic_example["note"][:800], "…")

original name : Harvey D'Amore
synthetic name: Kilby Orduna

**Medical Note**

**Patient Name:** Kilby Orduna  
**Date of Birth:** September 14, 1964  
**Medical Record Number:** UR-JX-5085941  
**Date of Encounter:** October 15, 2023  
**Attending Dentist:** Dr. Madalene Lenhoff  
**Location:** Gold Spur Systems, 341 Suburban Track, Springfield, IL  

**Reason for Visit:**  
Mr. Kilby Orduna presents for a routine check-up and management of gingivitis.

**Chief Complaint:**  
The patient reports experiencing swollen and tender gums over the past few weeks, with occasional bleeding during brushing and persistent bad breath.

**History of Present Illness:**  
Mr. Orduna is a 59-year-old male who has been experiencing gingival tenderness and bleeding upon brushing for approximately three weeks. He reports that bleeding is most notable when flossing an …


Now the full training split, with a distinct seed per record — synthetic identities vary across the dataset but stay consistent within each record. (Synthea patients have multiple encounters, so the same real patient appears in many notes; with per-record seeds each of their encounters gets a *different* synthetic identity. The extraction task only needs consistency within a record, and breaking linkability across records is a privacy bonus.)

Only the training split gets de-identified, because it's the thing that leaves the boundary. Evaluation happens later against the *real* validation notes, which stay in your environment — the same traffic the deployed model would see in production.

De-identifying ~1,000 records takes a few minutes; the result is cached to disk so re-running the notebook doesn't repeat the API calls.

In [8]:
SYNTH_CACHE = DATA_DIR / "synthetic_train.jsonl"

def synthesize_record(args, retries=3):
    i, record = args
    for attempt in range(retries):  # ~1k API calls; ride out transient network errors
        try:
            response = textual.redact_json(record, random_seed=i, **DEID_CONFIG)
            break
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)
    synthetic = json.loads(response.redacted_text)
    # redact_json drops empty fields ([] / null) from the reconstructed JSON.
    # Restore them so the training labels keep the full schema. Empty values
    # carry no PHI, which the assert enforces.
    for key, value in record["encounter_data"].items():
        if key not in synthetic["encounter_data"]:
            assert not value, f"non-empty field {key!r} went missing during synthesis"
            synthetic["encounter_data"][key] = value
    return synthetic

if SYNTH_CACHE.exists():
    synthetic_train = [json.loads(line) for line in SYNTH_CACHE.open()]
    print(f"loaded {len(synthetic_train)} cached records")
else:
    with cf.ThreadPoolExecutor(max_workers=6) as ex:
        synthetic_train = list(tqdm(
            ex.map(synthesize_record, enumerate(ds["train"])),
            total=len(ds["train"]),
        ))
    with SYNTH_CACHE.open("w") as f:
        for record in synthetic_train:
            f.write(json.dumps(record) + "\n")

loaded 1028 cached records


A quick audit before we call this safe: did any original identity survive into its synthetic counterpart?

In [9]:
original_names = [r["encounter_data"]["name"] for r in ds["train"]]
synthetic_names = [r["encounter_data"]["name"] for r in synthetic_train]

survivors = [
    orig
    for orig, synth in zip(original_names, synthetic_train)
    if orig in synth["note"] or orig == synth["encounter_data"]["name"]
]
key_mismatches = sum(
    orig["encounter_data"].keys() != synth["encounter_data"].keys()
    for orig, synth in zip(ds["train"], synthetic_train)
)

print("records where the original name survived:", len(survivors))
print("full-name overlap between real and synthetic sets:",
      len(set(original_names) & set(synthetic_names)))
print("records with missing/extra label fields:", key_mismatches)

pd.DataFrame({"original": original_names[:8], "synthetic": synthetic_names[:8]})

records where the original name survived: 0
full-name overlap between real and synthetic sets: 0
records with missing/extra label fields: 0


,original,synthetic
0,Harvey D'Amore,Aniken Mielke
1,Harvey D'Amore,Lakeland Blackston
2,Harvey D'Amore,Abdulwadud Manire
3,Harvey D'Amore,Kenyea Avilez
4,Kathern Laurence Nader,Drayah Sahel Rury
5,Mafalda Amy Wisozk,Bahia Estacia Durett
6,Mafalda Amy Wisozk,Michaelann Suma Sade
7,Mafalda Amy Wisozk,Cassia Haliegh Foard


## 3. Define the extraction task

The target is one typed record per encounter. `race` and `gender` are constrained to HL7 value sets ([US Core Race](https://www.hl7.org/fhir/us/core/StructureDefinition-us-core-race.html), [Gender Identity](https://terminology.hl7.org/ValueSet-gender-identity.html)); the remaining fields are free-form but formatted per the rules in the system prompt. To be clear about scope: this is not a full FHIR resource — it's the structured, vocabulary-controlled layer you'd map into one.

The Pydantic model below does double duty: it documents the schema inside the system prompt, and at evaluation time it validates model outputs.

In [10]:
from enum import Enum
from typing import List, Optional

from pydantic import BaseModel

class GenderIdentityEnum(str, Enum):
    """HL7 THO 'Gender Identity' value set."""
    FEMALE = "female"
    MALE = "male"
    NON_BINARY = "non-binary"
    ASKED_DECLINED = "asked-declined"
    UNKNOWN = "unknown"

class FhirUsCoreRaceEnum(str, Enum):
    """HL7 'US Core Race' value set."""
    WHITE = "white"
    ASIAN = "asian"
    AFRICAN_AMERICAN = "black or african american"
    NATIVE_HAWAIIAN = "native hawaiian or other pacific islander"
    AMERICAN_INDIAN = "american indian or alaska native"
    UNKNOWN = "unknown"

class EncounterData(BaseModel):
    name_given: str              # patient's first given name
    name_family: str             # patient's family name (surname)
    age: int                     # age in years at encounter
    race: FhirUsCoreRaceEnum
    gender: GenderIdentityEnum
    reason: Optional[str]        # short diagnosis/complaint, title case
    procedures: List[str]        # distinct procedures, sentence case
    medications: List[str]       # generic drug names, [] if none
    encounter_type: str          # e.g. "encounter for check up"

In [11]:
SYSTEM_PROMPT = """You are Clinical-Note-Extractor v1.
Your sole task is to read a free-text clinical note and emit only a compact JSON object that validates against the following Pydantic schema:

```python
from typing import List, Optional
from enum import Enum
from pydantic import BaseModel

class GenderIdentityEnum(str, Enum):          # HL7 THO "Gender Identity"
    FEMALE = "female"
    MALE = "male"
    NON_BINARY = "non-binary"
    ASKED_DECLINED = "asked-declined"
    UNKNOWN = "unknown"

class FhirUsCoreRaceEnum(str, Enum):          # HL7 US Core Race
    WHITE = "white"
    ASIAN = "asian"
    AFRICAN_AMERICAN = "black or african american"
    NATIVE_HAWAIIAN = "native hawaiian or other pacific islander"
    AMERICAN_INDIAN = "american indian or alaska native"
    UNKNOWN = "unknown"

class EncounterData(BaseModel):
    name_given: str              # Patient's first given name only
    name_family: str             # Patient's family name (surname)
    age: int                     # Age in years at encounter
    race: FhirUsCoreRaceEnum     # One of the enum literals above
    gender: GenderIdentityEnum   # One of the enum literals above
    reason: Optional[str]        # Short (< 10 words) diagnosis/complaint, title-case
    procedures: List[str]        # List of distinct procedures in sentence case
    medications: List[str]       # Generic drug names or [] if none
    encounter_type: str          # SNOMED-style phrase, kebab or snake allowed
```

Extraction rules

Names – name_given is the patient's first given name only (exclude middle names); name_family is the surname exactly as written in the note.

Age – derive from "Date of Birth" and "Date of Encounter" when both are present; else use explicit age statements (e.g. "59-year-old").

Race & gender – map author wording to the exact enum literal. If absent or ambiguous, choose "unknown" (or "asked-declined" if the note explicitly says so).

Reason – the primary diagnosis or chief complaint, ≤ 10 words, Title-Case (e.g. "Gingivitis").

Procedures – every billed or documented procedure, sentence-case, deduplicated, stripped of trailing punctuation.

Medications – all meds started, stopped, or continued; generic names only; do not infer drugs not mentioned. Empty list if none.

encounter_type – one concise phrase such as "encounter for check up" or "emergency department visit".

Output format – return only the JSON, no Markdown fence, no commentary. Keys and enum values must be double-quoted.

If any required field truly cannot be filled, substitute the appropriate unknown literal (null only for reason).
Return nothing but the schema-compliant JSON."""

Format each record as a chat conversation — Fireworks fine-tunes on JSONL where each line is `{"messages": [...]}`, the same shape as a chat completion request. The assistant target is projected through the Pydantic schema, so the training labels contain exactly the fields the model must produce. We write one file per training arm.

In [12]:
def to_messages(record):
    target = EncounterData.model_validate(record["encounter_data"]).model_dump(mode="json")
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": record["note"]},
            {"role": "assistant", "content": json.dumps(target)},
        ]
    }

def write_jsonl(path, records):
    with open(path, "w") as f:
        for record in records:
            f.write(json.dumps(to_messages(record)) + "\n")
    print(f"{path}: {len(records)} examples")

write_jsonl(DATA_DIR / "train_real.jsonl", ds["train"])
write_jsonl(DATA_DIR / "train_synthetic.jsonl", synthetic_train)

data/train_real.jsonl: 1028 examples
data/train_synthetic.jsonl: 1028 examples


## 4. Fine-tune on Fireworks

Two identical LoRA jobs on Llama 3.2 3B Instruct, differing only in their training data.

Worth stating plainly: **in a real deployment, only `train_synthetic.jsonl` would ever be uploaded.** We upload the real training file here purely to build the baseline that measures what synthesis costs us.

The helpers below are idempotent — datasets and jobs are looked up by id first, so re-running the notebook won't duplicate work or retrain.

In [13]:
def ensure_dataset(dataset_id: str, path: Path) -> str:
    """Upload a JSONL file as a Fireworks dataset, skipping if it already exists."""
    try:
        dataset = fw.datasets.get(dataset_id)
    except NotFoundError:
        example_count = sum(1 for _ in open(path))
        fw.datasets.create(
            dataset_id=dataset_id,
            dataset={
                "display_name": dataset_id,
                "user_uploaded": {},
                "example_count": str(example_count),
            },
        )
        with open(path, "rb") as f:
            fw.datasets.upload(dataset_id, file=f)
        dataset = fw.datasets.get(dataset_id)
    print(f"{dataset.name}: {dataset.state}")
    return dataset.name

real_dataset = ensure_dataset("clinical-notes-train-real", DATA_DIR / "train_real.jsonl")
synth_dataset = ensure_dataset("clinical-notes-train-synthetic", DATA_DIR / "train_synthetic.jsonl")

accounts/your-account-id/datasets/clinical-notes-train-real: READY
accounts/your-account-id/datasets/clinical-notes-train-synthetic: READY


In [14]:
def ensure_sft_job(job_id: str, dataset_name: str):
    try:
        job = fw.supervised_fine_tuning_jobs.get(job_id)
        print(f"{job_id}: already exists ({job.state})")
    except NotFoundError:
        job = fw.supervised_fine_tuning_jobs.create(
            supervised_fine_tuning_job_id=job_id,
            display_name=job_id,
            base_model=BASE_MODEL,
            dataset=dataset_name,
            output_model=f"accounts/{FIREWORKS_ACCOUNT}/models/{job_id}",
            epochs=2,
            lora_rank=8,
        )
        print(f"{job_id}: created ({job.state})")
    return job

# Tip: model ids are not reusable after deletion (they tombstone) — pick
# fresh ids if you retrain from scratch.
JOB_IDS = {
    "synthetic": "clinical-note-extractor-synthetic",
    "real": "clinical-note-extractor-real",
}

ensure_sft_job(JOB_IDS["synthetic"], synth_dataset)
ensure_sft_job(JOB_IDS["real"], real_dataset);

clinical-note-extractor-synthetic: already exists (JOB_STATE_COMPLETED)
clinical-note-extractor-real: already exists (JOB_STATE_COMPLETED)


Wait for both jobs to finish (roughly 10–20 minutes each at this dataset size; they run concurrently). Progress, loss curves, and sample renders are also visible in the [Fireworks dashboard](https://app.fireworks.ai/dashboard/fine-tuning).

In [15]:
TERMINAL_STATES = {
    "JOB_STATE_COMPLETED",
    "JOB_STATE_EARLY_STOPPED",
    "JOB_STATE_FAILED",
    "JOB_STATE_CANCELLED",
    "JOB_STATE_EXPIRED",
}

while True:
    jobs = {arm: fw.supervised_fine_tuning_jobs.get(job_id) for arm, job_id in JOB_IDS.items()}
    states = {arm: job.state for arm, job in jobs.items()}
    print(time.strftime("%H:%M:%S"), states)
    if all(state in TERMINAL_STATES for state in states.values()):
        break
    time.sleep(60)

for arm, job in jobs.items():
    assert job.state in ("JOB_STATE_COMPLETED", "JOB_STATE_EARLY_STOPPED"), f"{arm}: {job.status}"

SYNTH_MODEL = jobs["synthetic"].output_model
REAL_MODEL = jobs["real"].output_model
print("\nfine-tuned models:")
print(" ", SYNTH_MODEL)
print(" ", REAL_MODEL)

14:10:53 {'synthetic': 'JOB_STATE_COMPLETED', 'real': 'JOB_STATE_COMPLETED'}

fine-tuned models:
  accounts/your-account-id/models/clinical-note-extractor-synthetic
  accounts/your-account-id/models/clinical-note-extractor-real


## 5. Deploy and evaluate on Fireworks

Time to measure. The evaluation set is the **real** validation notes — real notes are what the model faces in production, and the synthetic-trained model has never seen one.

One dedicated deployment serves all three contenders: fine-tuned models on Fireworks run from on-demand deployments, and a single deployment of the base model with **LoRA addons** enabled hosts both adapters, with requests routed per call via the `#deployment` suffix. The deployment bills per GPU-second while it exists — the cleanup cell in section 6 tears it down.

In [16]:
# Note: deployment ids (like model ids) tombstone after deletion — reviving a
# deleted deployment can leave it READY but unable to serve. Use a fresh id.
DEPLOYMENT_ID = "clinical-note-extractor"

try:
    deployment = fw.deployments.get(DEPLOYMENT_ID)
except NotFoundError:
    deployment = fw.deployments.create(
        deployment_id=DEPLOYMENT_ID,
        display_name=DEPLOYMENT_ID,
        base_model=BASE_MODEL,
        enable_addons=True,
        accelerator_type="NVIDIA_B200_180GB",  # current Fireworks fleet; fits the 3B base + LoRA addons easily
    )

while deployment.state in ("CREATING", "UPDATING"):
    print(deployment.state, "…")
    time.sleep(15)
    deployment = fw.deployments.get(DEPLOYMENT_ID)

assert deployment.state == "READY", (deployment.state, deployment.status)
print("deployment ready:", deployment.name)

deployment ready: accounts/your-account-id/deployments/clinical-note-extractor


In [17]:
def wait_for_model(model_name: str):
    # freshly fine-tuned models finish uploading shortly after the job completes
    model_id = model_name.split("/")[-1]
    model = fw.models.get(model_id)
    while model.state == "UPLOADING":
        time.sleep(15)
        model = fw.models.get(model_id)
    assert model.state == "READY", (model_name, model.state)

def ensure_lora_loaded(model_name: str):
    wait_for_model(model_name)
    loaded = next(
        (dm for dm in fw.lora.list()
         if dm.model == model_name and dm.deployment == deployment.name),
        None,
    )
    if loaded is None:
        loaded = fw.lora.load(model=model_name, deployment=deployment.name)
    while loaded.state == "DEPLOYING":
        time.sleep(10)
        loaded = fw.lora.get(loaded.name.split("/")[-1])
    assert loaded.state == "DEPLOYED", loaded.state
    print(f"{model_name}: {loaded.state}")

ensure_lora_loaded(SYNTH_MODEL)
ensure_lora_loaded(REAL_MODEL)

# "<model>#<deployment>" routes a request to a specific model on our deployment
CONTENDERS = {
    "base (zero-shot)": f"{BASE_MODEL}#{deployment.name}",
    "fine-tuned on synthetic": f"{SYNTH_MODEL}#{deployment.name}",
    "fine-tuned on real": f"{REAL_MODEL}#{deployment.name}",
}

accounts/your-account-id/models/clinical-note-extractor-synthetic: DEPLOYED
accounts/your-account-id/models/clinical-note-extractor-real: DEPLOYED


In [18]:
EVAL_N = 150
eval_records = ds["validation"].select(range(EVAL_N))

SCALAR_FIELDS = ["name_given", "name_family", "age", "gender", "race", "reason", "encounter_type"]
LIST_FIELDS = ["procedures", "medications"]

def extract(model_ref: str, note: str) -> dict:
    response = fw.chat.completions.create(
        model=model_ref,
        temperature=0,
        max_tokens=768,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": note},
        ],
    )
    text = response.choices[0].message.content.strip()
    if text.startswith("```"):  # base model sometimes ignores the no-fence rule
        text = text.strip("`").removeprefix("json").strip()
    return json.loads(text)

def norm(value):
    return str(value).strip().lower() if value is not None else None

def set_f1(pred: list, true: list) -> float:
    pred, true = {norm(x) for x in pred}, {norm(x) for x in true}
    if not pred and not true:
        return 1.0
    overlap = len(pred & true)
    if not overlap:
        return 0.0
    precision, recall = overlap / len(pred), overlap / len(true)
    return 2 * precision * recall / (precision + recall)

def score_one(model_ref: str, record: dict) -> dict:
    true = record["encounter_data"]
    try:
        pred = EncounterData.model_validate(extract(model_ref, record["note"]))
        pred = pred.model_dump(mode="json")
    except Exception:
        return {"schema-valid": 0.0, **{f: 0.0 for f in SCALAR_FIELDS + LIST_FIELDS}}
    row = {"schema-valid": 1.0}
    for field in SCALAR_FIELDS:
        row[field] = float(norm(pred.get(field)) == norm(true[field]))
    for field in LIST_FIELDS:
        row[field] = set_f1(pred.get(field) or [], true[field])
    return row

def evaluate(model_ref: str) -> pd.Series:
    with cf.ThreadPoolExecutor(max_workers=8) as ex:
        rows = list(tqdm(
            ex.map(lambda record: score_one(model_ref, record), eval_records),
            total=EVAL_N,
        ))
    return pd.DataFrame(rows).mean()

In [19]:
results = {}
for arm, model_ref in CONTENDERS.items():
    print(arm)
    results[arm] = evaluate(model_ref)

results = pd.DataFrame(results).T
(DATA_DIR / "eval_results.json").write_text(results.T.to_json(indent=1))
results.round(3)

base (zero-shot)


fine-tuned on synthetic


fine-tuned on real


,schema-valid,name_given,name_family,age,gender,race,reason,encounter_type,procedures,medications
base (zero-shot),0.293,0.140,0.287,0.293,0.287,0.260,0.140,0.093,0.124,0.119
fine-tuned on synthetic,0.993,0.993,0.993,0.993,0.993,0.967,0.893,0.947,0.876,0.955
fine-tuned on real,0.973,0.973,0.973,0.973,0.973,0.953,0.880,0.927,0.859,0.938


### Reading the results

Two comparisons matter, and the table settles both:

- **Base vs. either fine-tune** — the case for fine-tuning. The un-tuned 3B model produces schema-valid JSON less than a third of the time and lands near zero on the formatting-sensitive fields (`encounter_type` 0.09, `medications` 0.12). Both fine-tunes clear 97% schema validity and 0.86 on every field.
- **Synthetic vs. real** — the point of the notebook. The synthetic-trained model matches the real-trained one everywhere; in this run it actually came out slightly *ahead* on most fields (0.99 vs 0.97 schema validity — differences of this size are within run-to-run noise). Both extract `name_given` and `name_family` at 0.97+, despite the synthetic-trained model having been trained entirely on invented identities: it learned *where names live in a clinical note*, not which names exist. De-identification cost nothing measurable.

And one difference the table can't show: the real-trained model was optimized to emit 1,028 actual patient identities — its weights are a PHI liability, in principle promptable into regurgitating real names. The synthetic-trained model *cannot* leak a real identity. It has never seen one.

## 6. Clean up

Delete the deployment to stop GPU billing (skip this if you're keeping the endpoint up for real traffic). The datasets and fine-tuned models remain in your account, ready to redeploy.

In [20]:
for deployed in list(fw.lora.list()):
    if deployed.deployment == deployment.name:
        fw.lora.unload(deployed.name.split("/")[-1])
        print("unloaded", deployed.model)

# ignore_checks: Fireworks guards against deleting deployments with recent
# traffic — here that traffic was our own evaluation, so it's safe to skip
fw.deployments.delete(DEPLOYMENT_ID, ignore_checks=True)
print("deployment deleted — billing stopped")

unloaded accounts/your-account-id/models/clinical-note-extractor-real
unloaded accounts/your-account-id/models/clinical-note-extractor-synthetic
deployment deleted — billing stopped


## Takeaway

We fine-tuned a clinical-note extractor without any PHI leaving the compliance boundary: Tonic Textual synthesized the identities in the training notes (consistently within each record), and Fireworks trained both models and serves them from a single multi-LoRA deployment. Evaluation on real held-out notes showed the synthetic-trained model performing on par with — this run, marginally better than — one trained on the original data, at more than triple the un-tuned base model's schema validity.

To adapt this to your own notes, swap in your data loader and revisit two knobs: `PHI_LABELS` (which entity types get synthesized) and the extraction schema/system prompt. Everything else — the consistency-preserving `redact_json` step, the two-arm training harness, the leakage audit, the evaluation — carries over unchanged.